# 01 · Exploratory Data Analysis — ASL Alphabet

**Person A.** Goal: understand class balance, image properties, and — most importantly — *demonstrate the single-signer trap* so the whole team internalises why we never random-split.

Run from the repo root so `import src` works.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, cv2, matplotlib.pyplot as plt
from src.data import CLASSES, class_counts, list_class_files, frame_range_split

ROOT = '../data/raw/asl_alphabet_train'  # adjust if needed

## Class balance
Expect ~3,000 images per class, 29 classes.

In [ ]:
counts = class_counts(ROOT)
plt.figure(figsize=(12,4))
plt.bar(list(counts.keys()), list(counts.values()))
plt.xticks(rotation=90); plt.ylabel('images'); plt.title('Class balance')
plt.tight_layout(); plt.savefig('../docs/figures/class_balance.png', dpi=150)
print('total images:', sum(counts.values()))

## The single-signer trap (show it, don't just say it)
Plot consecutive frames for one class. They should look near-identical — which is exactly why a random split leaks.

In [ ]:
files = list_class_files(ROOT, 'A')[:6]
fig, axes = plt.subplots(1, 6, figsize=(15,3))
for ax, f in zip(axes, files):
    ax.imshow(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)); ax.axis('off')
    ax.set_title(os.path.basename(f), fontsize=8)
plt.suptitle('Consecutive frames of class A — note how similar they are'); plt.show()

## The leakage-safe split
Build the frame-range split and confirm train precedes val within each class. (The formal proof lives in `tests/test_split_leakage.py`.)

In [ ]:
m = frame_range_split(ROOT, train_frac=0.8)
print('train', len(m.train), 'val', len(m.val))
train_paths = {s.path for s in m.train}; val_paths = {s.path for s in m.val}
print('overlap (must be 0):', len(train_paths & val_paths))

## TODO
- Image size / channel sanity (all 200x200 RGB?).
- Inspect the hand crops produced by `src.crop` on a few training images.
- Note candidate confusions to watch: M/N/S/T, A/E, K/V.